In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
np.random.seed(SEED)


In [2]:
DATA_PATH = "merged_df_cleaned.csv"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]

if "survived" not in df.columns:
    raise ValueError("Expected target column 'survived' was not found.")

df = df.dropna(subset=["survived"]).copy()
df["survived"] = df["survived"].astype(int)

print("Dataset shape:", df.shape)
print("Target distribution:")
print(df["survived"].value_counts(normalize=True).rename("ratio"))


Dataset shape: (125922, 38)
Target distribution:
survived
1    0.794555
0    0.205445
Name: ratio, dtype: float64


In [3]:
identifier_cols = [c for c in ["store_id", "name", "address", "genres_list"] if c in df.columns]

X = df.drop(columns=["survived"] + identifier_cols)
y = df["survived"]

categorical_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object", "category", "string"]).columns.tolist()

print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Numeric features: 26
Categorical features: 7
Train shape: (100737, 33)
Test shape: (25185, 33)


In [4]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=30)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("cat", categorical_pipe, categorical_cols),
], remainder="drop")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)


In [5]:
searches = {
    "logistic_regression": GridSearchCV(
        estimator=Pipeline([
            ("prep", preprocessor),
            ("model", LogisticRegression(
                max_iter=1500,
                class_weight="balanced",
                solver="liblinear",
                random_state=SEED,
            )),
        ]),
        param_grid={"model__C": [0.2, 1.0, 5.0]},
        scoring="f1_macro",
        cv=cv,
        n_jobs=1,
        verbose=1,
    ),
    "random_forest": GridSearchCV(
        estimator=Pipeline([
            ("prep", preprocessor),
            ("model", RandomForestClassifier(
                random_state=SEED,
                n_jobs=1,
                class_weight="balanced_subsample",
            )),
        ]),
        param_grid={
            "model__n_estimators": [300],
            "model__max_depth": [None, 25],
            "model__min_samples_leaf": [1, 3],
        },
        scoring="f1_macro",
        cv=cv,
        n_jobs=1,
        verbose=1,
    ),
    "extra_trees": GridSearchCV(
        estimator=Pipeline([
            ("prep", preprocessor),
            ("model", ExtraTreesClassifier(
                random_state=SEED,
                n_jobs=1,
                class_weight="balanced",
            )),
        ]),
        param_grid={
            "model__n_estimators": [400],
            "model__max_depth": [None, 30],
            "model__min_samples_leaf": [1, 3],
        },
        scoring="f1_macro",
        cv=cv,
        n_jobs=1,
        verbose=1,
    ),
}

results = []
best_name = None
best_search = None
best_cv_score = -np.inf

for name, search in searches.items():
    print(f"\n=== Training: {name} ===")
    search.fit(X_train, y_train)
    score = float(search.best_score_)
    results.append({
        "model": name,
        "best_cv_f1_macro": score,
        "best_params": search.best_params_,
    })
    print(f"Best CV f1_macro ({name}): {score:.4f}")
    print(f"Best params ({name}): {search.best_params_}")

    if score > best_cv_score:
        best_cv_score = score
        best_name = name
        best_search = search

results_df = pd.DataFrame(results).sort_values("best_cv_f1_macro", ascending=False)
print("\nModel ranking by CV f1_macro:")
print(results_df[["model", "best_cv_f1_macro"]])
print(f"\nSelected model: {best_name} (CV f1_macro={best_cv_score:.4f})")



=== Training: logistic_regression ===
Fitting 5 folds for each of 3 candidates, totalling 15 fits
Best CV f1_macro (logistic_regression): 0.5485
Best params (logistic_regression): {'model__C': 0.2}

=== Training: random_forest ===
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best CV f1_macro (random_forest): 0.5972
Best params (random_forest): {'model__max_depth': None, 'model__min_samples_leaf': 3, 'model__n_estimators': 300}

=== Training: extra_trees ===
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best CV f1_macro (extra_trees): 0.5800
Best params (extra_trees): {'model__max_depth': None, 'model__min_samples_leaf': 3, 'model__n_estimators': 400}

Model ranking by CV f1_macro:
                 model  best_cv_f1_macro
1        random_forest          0.597220
2          extra_trees          0.579978
0  logistic_regression          0.548526

Selected model: random_forest (CV f1_macro=0.5972)


In [6]:
best_model = best_search.best_estimator_

oof_proba = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=1,
)[:, 1]

thresholds = np.linspace(0.05, 0.95, 181)
macro_f1_by_threshold = [
    f1_score(y_train, (oof_proba >= t).astype(int), average="macro") for t in thresholds
]
best_threshold = float(thresholds[int(np.argmax(macro_f1_by_threshold))])

print("Best threshold (OOF, macro F1):", round(best_threshold, 4))
print("OOF macro F1 at best threshold:", round(float(np.max(macro_f1_by_threshold)), 4))


Best threshold (OOF, macro F1): 0.485
OOF macro F1 at best threshold: 0.5988


In [7]:
test_proba = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

test_f1_macro = f1_score(y_test, test_pred, average="macro")
test_f1_binary = f1_score(y_test, test_pred)
test_accuracy = accuracy_score(y_test, test_pred)

print("Selected model:", best_name)
print("Estimated holdout test f1_macro:", round(float(test_f1_macro), 4))
print("Estimated holdout test f1 (positive class):", round(float(test_f1_binary), 4))
print("Estimated holdout test accuracy:", round(float(test_accuracy), 4))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification report:")
print(classification_report(y_test, test_pred, digits=4))


Selected model: random_forest
Estimated holdout test f1_macro: 0.5977
Estimated holdout test f1 (positive class): 0.8335
Estimated holdout test accuracy: 0.7359

Confusion matrix:
[[ 1886  3288]
 [ 3364 16647]]

Classification report:
              precision    recall  f1-score   support

           0     0.3592    0.3645    0.3619      5174
           1     0.8351    0.8319    0.8335     20011

    accuracy                         0.7359     25185
   macro avg     0.5972    0.5982    0.5977     25185
weighted avg     0.7373    0.7359    0.7366     25185

